In [1]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import pandas as pd
import numpy as np

# ================================
# 2. LOAD DATA
# ================================
df = pd.read_excel("online_retail_II.xlsx")

# ================================
# 3. BASIC CLEANING
# ================================
df.dropna(subset=['Customer ID'], inplace=True)

# Convert date column
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Create Revenue column
df['Revenue'] = df['Quantity'] * df['Price']

# ================================
# 4. CREATE COHORT (FIRST PURCHASE MONTH)
# ================================
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

# First purchase month per customer
df['CohortMonth'] = df.groupby('Customer ID')['InvoiceMonth'].transform('min')

# ================================
# 5. CREATE COHORT INDEX (MONTH DIFFERENCE)
# ================================
df['CohortIndex'] = (df['InvoiceMonth'] - df['CohortMonth']).apply(lambda x: x.n)

# ================================
# 6. CREATE RETENTION TABLE
# ================================
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['Customer ID'].nunique().reset_index()

# Pivot table
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='Customer ID')

# ================================
# 7. CALCULATE RETENTION RATE
# ================================
retention_matrix = cohort_pivot.divide(cohort_pivot[0], axis=0)

# ================================
# 8. VALIDATION CHECKS
# ================================

# Check 1: First column should be 1 (100%)
print("Check 1: First column values (should be 1)")
print(retention_matrix[0].head())

# Check 2: Values should be between 0 and 1
print("\nCheck 2: Retention values range")
print(retention_matrix.describe())

# Check 3: Missing values
print("\nCheck 3: Missing values in matrix")
print(retention_matrix.isnull().sum().sum())

# ================================
# 9. DISPLAY RESULT
# ================================
print("\nRetention Matrix:")
print(retention_matrix.round(3))

Check 1: First column values (should be 1)
CohortMonth
2009-12    1.0
2010-01    1.0
2010-02    1.0
2010-03    1.0
2010-04    1.0
Freq: M, Name: 0, dtype: float64

Check 2: Retention values range
CohortIndex    0          1          2          3         4         5   \
count        13.0  12.000000  11.000000  10.000000  9.000000  8.000000   
mean          1.0   0.243333   0.235006   0.255312  0.243249  0.247323   
std           0.0   0.066389   0.069712   0.091260  0.076821  0.080579   
min           1.0   0.118012   0.102902   0.115702  0.126582  0.114754   
25%           1.0   0.213623   0.198169   0.195800  0.188976  0.206910   
50%           1.0   0.224257   0.225895   0.260196  0.230483  0.245042   
75%           1.0   0.296705   0.281181   0.304184  0.279188  0.284865   
max           1.0   0.375120   0.342584   0.427751  0.392344  0.390431   

CohortIndex        6         7         8         9         10        11  \
count        7.000000  6.000000  5.000000  4.000000  3.000000 